# 🩺 Huấn luyện UNetBase (Classic CNN) trên Google Colab

Quy trình chuẩn hóa toàn diện:
1. **Kiểm tra phần cứng GPU** (`!nvidia-smi`)
2. **Đồng bộ mã nguồn từ GitHub**
3. **Cài đặt các thư viện y tế phụ thuộc** (SimpleITK, etc.)
4. **Mount Google Drive**
5. **Giải nén dữ liệu từ Google Drive vào SSD cục bộ** (có tùy chọn số lượng ca bệnh nhân `NUM_PATIENTS`)
6. **Huấn luyện mô hình với đầy đủ tham số tùy chỉnh** (Features, Activation, Loss, LR, Workers, Resume...)
7. **Đánh giá 3D cấp độ bệnh nhân** trên tập Validation
8. **Lưu kết quả & Checkpoints vào Google Drive theo từng thư mục phiên chạy kèm bản báo cáo Markdown/JSON** (tự động append nếu Resume)

In [ ]:
# 1. Kiểm tra phần cứng và GPU khả dụng
!nvidia-smi

In [ ]:
# 2. Tải và đồng bộ mã nguồn từ GitHub
import os, sys

GITHUB_REPO = 'https://github.com/NguyenMinhTri24072005/CT208E_LIDC_DuneAI_MambaVision.git'
BRANCH = 'main'  # Nhánh mặc định trên GitHub (đổi nếu bạn dùng nhánh khác)
REPO_DIR = '/content/CT208E_LIDC_DuneAI_MambaVision'

if os.path.exists('unet_base') and os.path.exists('common'):
    print(f'✅ Đang chạy trực tiếp trong thư mục dự án: {os.getcwd()}')
else:
    if not os.path.exists(REPO_DIR):
        print(f'📥 Đang clone repository từ GitHub (nhánh {BRANCH})...')
        !git clone -b {BRANCH} {GITHUB_REPO} {REPO_DIR}
    else:
        print(f'🔄 Đang đồng bộ mã nguồn mới nhất từ GitHub...')
        %cd {REPO_DIR}
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    %cd {REPO_DIR}

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'🚀 Thư mục làm việc hiện tại: {os.getcwd()}')

In [ ]:
# 3. Cài đặt các thư viện phụ thuộc
!pip install -q SimpleITK pandas scipy opencv-python-headless pyyaml matplotlib

In [ ]:
# 4. Kết nối với Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 5. Chuẩn bị dữ liệu từ Google Drive vào SSD cục bộ (/content/data/cache_1mm)
# Thư mục GG_1 trên máy tính được mirror trực tiếp vào Google Drive.
# Mã sẽ tự động tìm kiếm file zip theo đúng cấu trúc mirror của bạn:
import os
from common.data_pipeline.runtime_data import run_colab

DRIVE_ZIP_CANDIDATES = [
    # 1. Mirror trực tiếp thư mục GG_1 vào MyDrive:
    '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/cache_1mm.zip',
    # 2. Mirror có thư mục cha GG_1 trong MyDrive:
    '/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/cache_1mm.zip',
    # 3. File zip tại thư mục KAGGLE_UPLOAD:
    '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/KAGGLE_UPLOAD/cache_1mm.zip',
    # 4. File zip đặt ngay tại gốc MyDrive:
    '/content/drive/MyDrive/cache_1mm.zip',
]

DRIVE_ZIP_PATH = None
for cand in DRIVE_ZIP_CANDIDATES:
    if os.path.exists(cand):
        DRIVE_ZIP_PATH = cand
        print(f'🎯 Đã tìm thấy file zip: {DRIVE_ZIP_PATH}')
        break

if DRIVE_ZIP_PATH is None:
    print('⚠️ Chưa thấy ở các đường dẫn cố định, sẽ tự động dò tìm sâu dưới /content/drive...')
    DRIVE_ZIP_PATH = 'auto'

# NUM_PATIENTS:
# - 0   : Giải nén toàn bộ 1010 bệnh nhân (cần ~40-60GB SSD Colab).
# - > 0 : Chọn N bệnh nhân đầu tiên (vd: 50, 100, 200) để huấn luyện nhanh và tiết kiệm dung lượng đĩa.
NUM_PATIENTS = 0

cache_dir = run_colab(
    zip_path=DRIVE_ZIP_PATH,
    dest_dir='/content/data/cache_1mm',
    n_patients=NUM_PATIENTS,
)
print(f'✅ Dữ liệu sẵn sàng tại: {cache_dir}')

In [ ]:
# 6. Cấu hình chi tiết & Bắt đầu huấn luyện mô hình UNetBase
import os

# =============================================================================
# BẢNG THAM SỐ TÙY CHỈNH (Tùy chỉnh trực tiếp các giá trị bên dưới)
# =============================================================================
# 1. Kiến trúc mô hình UNetBase
FEATURES = 64                       # Số kênh đặc trưng cơ sở (mặc định 64: [64, 128, 256, 512])

# 2. Hàm kích hoạt (Activation)
# Lựa chọn: 'relu', 'leaky_relu', 'gelu', 'silu', 'mish', 'identity'
ACTIVATION = "relu"

# 3. Hàm mất mát (Loss Function)
# Lựa chọn: 'dice_bce', 'dice', 'bce', 'tversky', 'focal_tversky', 'focal'
LOSS = "dice_bce"
LOSS_PARAMS = '{"bce_weight": 0.5, "dice_weight": 0.5}'  # Chuỗi JSON tham số loss, hoặc để trống ''

# 4. Siêu tham số huấn luyện (Training Hyperparameters)
EPOCHS = 50                         # Tổng số epoch huấn luyện
BATCH_SIZE = 8                      # Kích thước batch
LR = 1e-4                           # Tốc độ học cơ sở (Learning rate)
WEIGHT_DECAY = 1e-4                 # Hệ số suy giảm trọng số AdamW
SCHEDULER = "cosine"                # Lựa chọn: 'cosine', 'plateau', 'step', 'none'
GRAD_CLIP = 1.0                     # Ngưỡng kẹp gradient tối đa (0 = tắt)
PATIENCE = 12                       # Số epoch chờ Early Stopping
WORKERS = 2                         # Số luồng nạp dữ liệu DataLoader (Colab khuyến nghị 2-4)

# 5. Quản lý phiên chạy & Khôi phục (Resume)
# RESUME: None (chạy mới từ đầu), 'auto' (tiếp tục phiên gần nhất bị ngắt), hoặc đường dẫn checkpoint/run cụ thể
RESUME = None                       # vd: 'auto' hoặc 'unet_base/runs/run_xxx/last_model.pth'
INIT_WEIGHTS = None                 # Nạp trọng số ban đầu nhưng tạo run mới (None nếu không dùng)
RUN_NAME = "unet_colab_run"         # Tên phiên chạy (để trống '' để tự sinh theo timestamp)
TAG = "colab_unet"                 # Thẻ phân loại thí nghiệm
NOTES = "Huấn luyện UNetBase trên Colab"
SEED = 42

# 6. Tối ưu phần cứng & Chuẩn bị dữ liệu
USE_AMP = True                      # Bật Automatic Mixed Precision fp16 tăng tốc
GPU_WINDOW = False                  # Chuyển HU int16 lên GPU chuẩn hóa cửa sổ
NO_LUNG_MASK = False                # True: Không áp mặt nạ phổi ngoài lát cắt
NO_AUGMENT = False                  # True: Tắt lật/xoay ngẫu nhiên
LIMIT_TRAIN = 0                     # Giới hạn số lát cắt train (0 = toàn bộ)
LIMIT_VAL = 0                       # Giới hạn số lát cắt val (0 = toàn bộ)

# =============================================================================
# TỔ HỢP LỆNH HUẤN LUYỆN
# =============================================================================
cmd_parts = [
    "python -m unet_base.train_main train",
    f"--cache-dir /content/data/cache_1mm",
    f"--features {FEATURES}",
    f"--activation {ACTIVATION}",
    f"--loss {LOSS}",
    f"--epochs {EPOCHS}",
    f"--batch-size {BATCH_SIZE}",
    f"--lr {LR}",
    f"--weight-decay {WEIGHT_DECAY}",
    f"--scheduler {SCHEDULER}",
    f"--grad-clip {GRAD_CLIP}",
    f"--patience {PATIENCE}",
    f"--workers {WORKERS}",
    f"--seed {SEED}",
]
if LOSS_PARAMS and LOSS_PARAMS.strip():
    cmd_parts.append(f"--loss-params '{LOSS_PARAMS.strip()}'")
if RESUME:
    cmd_parts.append(f"--resume {RESUME}")
if INIT_WEIGHTS:
    cmd_parts.append(f"--init-weights '{INIT_WEIGHTS}'")
if RUN_NAME:
    cmd_parts.append(f"--run-name '{RUN_NAME}'")
if TAG:
    cmd_parts.append(f"--tag '{TAG}'")
if NOTES:
    cmd_parts.append(f"--notes '{NOTES}'")
if not USE_AMP:
    cmd_parts.append("--no-amp")
if GPU_WINDOW:
    cmd_parts.append("--gpu-window")
if NO_LUNG_MASK:
    cmd_parts.append("--no-lung-mask")
if NO_AUGMENT:
    cmd_parts.append("--no-augment")
if LIMIT_TRAIN > 0:
    cmd_parts.append(f"--limit-train {LIMIT_TRAIN}")
if LIMIT_VAL > 0:
    cmd_parts.append(f"--limit-val {LIMIT_VAL}")

full_train_cmd = " \
    ".join(cmd_parts)
print("🚀 Đang khởi động tiến trình huấn luyện...")
print(f"Lệnh thực thi:\n{full_train_cmd}\n" + "=" * 80)
!{full_train_cmd}

In [ ]:
# 7. Đánh giá 3D cấp độ bệnh nhân trên tập Validation
!python -m unet_base.evaluate eval \
    --cache-dir /content/data/cache_1mm \
    --checkpoint checkpoints/best_model.pth \
    --split val

In [ ]:
# 8. Lưu kết quả và Checkpoint vào Google Drive theo từng thư mục phiên chạy
# Tự động tạo mới hoặc cập nhật nối thêm (append) file REPORT.md và JSON tổng kết
import datetime, glob, json, os, shutil
from pathlib import Path

# 1. Xác định thư mục mirror Google Drive
MIRROR_SOURCES_DIR = '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES'
if not os.path.exists(MIRROR_SOURCES_DIR):
    MIRROR_SOURCES_DIR = '/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES'
if not os.path.exists(MIRROR_SOURCES_DIR):
    MIRROR_SOURCES_DIR = '/content/drive/MyDrive/DuneAI_Backups'

# 2. Tìm thư mục phiên chạy gần nhất trong unet_base/runs
local_runs = sorted(glob.glob("unet_base/runs/*"), key=os.path.getmtime)
if local_runs:
    active_local_run = Path(local_runs[-1])
    run_folder_name = active_local_run.name
else:
    run_folder_name = f"run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    active_local_run = Path(f"unet_base/runs/{run_folder_name}")
    active_local_run.mkdir(parents=True, exist_ok=True)

# 3. Thư mục đích trên Google Drive
drive_run_dir = Path(MIRROR_SOURCES_DIR) / "unet_base" / "runs" / run_folder_name
drive_ckpt_dir = Path(MIRROR_SOURCES_DIR) / "checkpoints"
drive_run_dir.mkdir(parents=True, exist_ok=True)
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)

# 4. Trích xuất thông tin kết quả huấn luyện
now_str = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
best_val_dice = "N/A"
best_epoch = "N/A"
if (active_local_run / "train_log.csv").exists():
    try:
        import pandas as pd
        df_log = pd.read_csv(active_local_run / "train_log.csv")
        if "val_dice" in df_log.columns and len(df_log) > 0:
            best_idx = df_log["val_dice"].idxmax()
            best_val_dice = f"{df_log.loc[best_idx, 'val_dice']:.4f}"
            best_epoch = f"{int(df_log.loc[best_idx, 'epoch'])}"
    except Exception:
        pass

# 5. Soạn thảo / Cập nhật file báo cáo REPORT.md
report_md_path = drive_run_dir / "REPORT.md"
is_resumed_session = bool(RESUME) and report_md_path.exists()

if is_resumed_session:
    print(f"🔄 Đang ghi đè/nối thêm (append) thông tin phiên Resume vào: {report_md_path}")
    resume_block = f"""
---
### 🔄 Phiên Tiếp Tục Huấn Luyện (Resume Session) - {now_str}
- **Khôi phục từ**: `{RESUME}`
- **Số Epochs đích**: {EPOCHS} | **Batch Size**: {BATCH_SIZE} | **Learning Rate**: {LR}
- **Scheduler**: {SCHEDULER} | **Workers**: {WORKERS}
- **Best Val Dice hiện tại**: **{best_val_dice}** (tại Epoch {best_epoch})
- **Ghi chú**: {NOTES}
"""
    with open(report_md_path, "a", encoding="utf-8") as f:
        f.write(resume_block)
else:
    print(f"📝 Đang tạo mới file báo cáo tổng hợp: {report_md_path}")
    new_report = f"""# 📋 Báo Cáo Huấn Luyện UNetBase: {run_folder_name}

- **Thời gian khởi tạo**: {now_str}
- **Kiến trúc mô hình**: UNetBase (Classic CNN)
  - **Features**: {FEATURES} (Kênh cơ sở)
  - **Decoder Activation**: `{ACTIVATION}`
- **Hàm mất mát (Loss)**: `{LOSS}` (Tham số: `{LOSS_PARAMS}`)
- **Siêu tham số tối ưu**:
  - **Optimizer**: AdamW (LR: `{LR}`, Weight Decay: `{WEIGHT_DECAY}`)
  - **LR Scheduler**: `{SCHEDULER}`
  - **Gradient Clip**: `{GRAD_CLIP}` | **Patience (Early Stopping)**: `{PATIENCE}`
- **Cấu hình DataLoader**:
  - **Batch Size**: {BATCH_SIZE} | **Workers**: {WORKERS} | **Seed**: {SEED}
- **Kết quả huấn luyện**:
  - **Best Val Dice**: **{best_val_dice}** (tại Epoch {best_epoch})
- **Ghi chú**: {NOTES}
"""
    with open(report_md_path, "w", encoding="utf-8") as f:
        f.write(new_report)

# Sao lưu báo cáo vào thư mục cục bộ và lưu thêm bản JSON
shutil.copy2(report_md_path, active_local_run / "REPORT.md")
report_json_data = {
    "run_folder": run_folder_name,
    "timestamp": now_str,
    "arch": "unet_base",
    "features": FEATURES,
    "activation": ACTIVATION,
    "loss": LOSS,
    "loss_params": LOSS_PARAMS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "scheduler": SCHEDULER,
    "workers": WORKERS,
    "best_val_dice": best_val_dice,
    "best_epoch": best_epoch,
    "resume": RESUME,
    "notes": NOTES,
}
with open(drive_run_dir / "training_report.json", "w", encoding="utf-8") as f:
    json.dump(report_json_data, f, indent=2, ensure_ascii=False)
shutil.copy2(drive_run_dir / "training_report.json", active_local_run / "training_report.json")

# 6. Sao chép toàn bộ checkpoints và artifacts của phiên chạy vào Drive
print(f"🚀 Đang sao lưu toàn bộ files từ {active_local_run} sang {drive_run_dir}...")
!cp -ru {active_local_run}/* '{drive_run_dir}/' 2>/dev/null || true
!cp -ru checkpoints/* '{drive_ckpt_dir}/' 2>/dev/null || true

print("=" * 80)
print(f"✅ HOÀN TẤT SAO LƯU!")
print(f"📁 Thư mục phiên chạy tại Google Drive: {drive_run_dir}")
print(f"📄 Báo cáo chi tiết: {report_md_path}")
print(f"🎯 Google Drive Desktop sẽ tự động đồng bộ thư mục này về máy tính của bạn!")
print("=" * 80)